# Module 4: Accelerate LangGraph with NVIDIA

> Part of the **Modular Workshops** series. Standalone, ~10 min.

The `langchain-nvidia-langgraph` package drops two optimizations into any existing LangGraph app, no node rewrites required:

- **Parallel execution** — independent nodes are detected and run concurrently, eliminating sequential bottlenecks.
- **Speculative execution** — both branches of a conditional edge run at the same time; the wrong branch is discarded once routing resolves.

We build a small graph, observe sequential execution, then enable the optimization and measure the same graph completing faster.

**Docs:** <https://docs.langchain.com/oss/python/integrations/providers/nvidia>.


## Setup

Install `langchain-nvidia-langgraph` directly from the LangChain integration repo — it isn't on PyPI yet, so we point pip at the GitHub subdirectory that holds the package. No NVIDIA API key required; this package is a graph-optimization layer, not a model provider.

In [ ]:
!uv pip install -q "git+https://github.com/langchain-ai/langchain-nvidia.git#subdirectory=libs/langgraph"

## 1. A small graph with two independent nodes

Two nodes, each sleeping for one second. Neither depends on the other — they both read from the entry state and write into a shared `log` list (the `Annotated[..., add]` reducer merges concurrent writes).


In [ ]:
import time
from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import END, START


class State(TypedDict):
    log: Annotated[list[str], add]


def slow_node_a(state):
    time.sleep(1)
    return {"log": ["a done"]}


def slow_node_b(state):
    time.sleep(1)
    return {"log": ["b done"]}


## 2. Baseline — vanilla LangGraph

With standard `langgraph.StateGraph`, even independent nodes run serially when chained on a single path. Wire `a → b`, time it, expect ~2 seconds.


In [ ]:
from langgraph.graph import StateGraph

g = StateGraph(State)
g.add_node("a", slow_node_a)
g.add_node("b", slow_node_b)
g.add_edge(START, "a")
g.add_edge("a", "b")
g.add_edge("b", END)
baseline = g.compile()

t0 = time.perf_counter()
out = baseline.invoke({"log": []})
print(f"Baseline: {time.perf_counter() - t0:.2f}s  log={out['log']}")


## 3. Parallel execution with `langchain-nvidia-langgraph`

Same graph as §2 — `a → b` sequential — but compiled with `NvidiaStateGraph` and `OptimizationConfig(enable_parallel=True)`. The optimizer analyzes the node bodies, sees that `a` and `b` write to disjoint state, and rewrites the execution order to run them in parallel. No topology change required. Expect ~1 second.

In [ ]:
from langchain_nvidia_langgraph.graph import StateGraph as NvidiaStateGraph, OptimizationConfig

# Same graph shape as the baseline — sequential a → b — just compiled with the NVIDIA optimizer.
g = NvidiaStateGraph(State)
g.add_node("a", slow_node_a)
g.add_node("b", slow_node_b)
g.add_edge(START, "a")
g.add_edge("a", "b")
g.add_edge("b", END)
accelerated = g.compile(optimization=OptimizationConfig(enable_parallel=True))

t0 = time.perf_counter()
out = accelerated.invoke({"log": []})
print(f"Parallel: {time.perf_counter() - t0:.2f}s  log={out['log']}")

## 4. Speculative execution

A conditional edge picks one of two branches at runtime. With speculation on, both branches start running immediately; the discard happens once the router decides. Useful when branches are computationally expensive and the router decision is inexpensive relative to them.


In [ ]:
def router(state):
    time.sleep(1)  # routing decision takes a moment
    return "left"


def left(state):
    time.sleep(1)
    return {"log": ["left done"]}


def right(state):
    time.sleep(1)
    return {"log": ["right done"]}


g = NvidiaStateGraph(State)
g.add_node("router_node", lambda s: s)
g.add_node("left", left)
g.add_node("right", right)
g.add_edge(START, "router_node")
g.add_conditional_edges("router_node", router, {"left": "left", "right": "right"})
g.add_edge("left", END)
g.add_edge("right", END)
speculative = g.compile(
    optimization=OptimizationConfig(enable_parallel=True, enable_speculation=True),
)

t0 = time.perf_counter()
out = speculative.invoke({"log": []})
print(f"Speculative: {time.perf_counter() - t0:.2f}s  log={out['log']}")


## Recap

| Optimization | How | When it helps |
|---|---|---|
| **Parallel** | `OptimizationConfig(enable_parallel=True)` | Two or more nodes share no state dependencies. |
| **Speculative** | `OptimizationConfig(enable_parallel=True, enable_speculation=True)` | Conditional edges where branches are expensive and the router is cheap relative to the branches. Speculation rides on top of the parallel rewriter, so both flags must be enabled together. |

**Wrap an existing graph instead of swapping `StateGraph`:** `with_app_compile(graph).compile(optimization=...)` — same effect, no import changes.

**Decorators for explicit control** (when the optimizer can't infer safety from the graph alone): `@sequential` (force serial), `@depends_on("name")` (declare a dependency missing from edges), `@speculation_unsafe` (opt a node out of speculation). See the docs page for examples.
